In [ ]:
!pip3 install torch torchvision pillow

In [8]:
import os
import io
import time
import torch
import zipfile
import torchvision
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as transforms

from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Sistem berjalan menggunakan perangkat: {device}")

# ==========================================
# 1. PERSIAPAN DATA & AUGMENTASI
# ==========================================
aturan_latih = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

aturan_uji = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# --- BAGIAN A: Menyiapkan Data Bawaan PyTorch ---
print("\nMengunduh/Memuat Dataset CIFAR-10 Bawaan...")
data_latih = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=aturan_latih)
data_uji = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=aturan_uji)

penyuap_latih = DataLoader(data_latih, batch_size=32, shuffle=True)
penyuap_uji = DataLoader(data_uji, batch_size=32, shuffle=False)

# --- BAGIAN B: Menyiapkan Data Sumber Baru (Konsep Sama Seperti Bagian A) ---
print("\nMengunduh/Memuat Dataset Uji Baru...")

# Menggunakan konsep persis seperti Bagian A (Tanpa perlu urllib atau zipfile)
# Jika folder './sumber_data_berbeda' belum ada, PyTorch akan otomatis membuatnya
dataset_sumber_baru = torchvision.datasets.CIFAR10(
    root='./sumber_data_berbeda',
    train=False,           # False karena ini untuk dataset pengujian
    download=True,         # Otomatis mengunduh data jika folder kosong
    transform=aturan_uji
)

penyuap_uji_baru = DataLoader(dataset_sumber_baru, batch_size=32, shuffle=False)

print(f"🎉 Data berhasil dimuat! Total: {len(dataset_sumber_baru)} gambar siap dievaluasi.")

Sistem berjalan menggunakan perangkat: cpu

Mengunduh/Memuat Dataset CIFAR-10 Bawaan...

Mengunduh/Memuat Dataset Uji Baru...


100%|██████████| 170M/170M [00:01<00:00, 96.0MB/s]


🎉 Data berhasil dimuat! Total: 10000 gambar siap dievaluasi.


In [5]:
# ==========================================
# 2. ARSITEKTUR OTAK
# ==========================================
class Otak(nn.Module):
    def __init__(self):
        super().__init__()

        # BLOK 1
        self.mata1a = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1a = nn.BatchNorm2d(32)
        self.mata1b = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.bn1b = nn.BatchNorm2d(32)
        self.kacamata_pengecil1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.anti_hafal1 = nn.Dropout(p=0.2)

        # BLOK 2
        self.mata2a = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2a = nn.BatchNorm2d(64)
        self.mata2b = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2b = nn.BatchNorm2d(64)
        self.kacamata_pengecil2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.anti_hafal2 = nn.Dropout(p=0.3)

        # BLOK 3
        self.mata3a = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3a = nn.BatchNorm2d(128)
        self.mata3b = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn3b = nn.BatchNorm2d(128)
        self.kacamata_pengecil3 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.anti_hafal3 = nn.Dropout(p=0.4)

        # OTAK KEPUTUSAN FINAL
        # Menggunakan AdaptiveAvgPool menjadikan input linear tetap 128, berapapun ukuran gambar awal
        self.pooling_final = nn.AdaptiveAvgPool2d((1, 1))
        self.otak_utama = nn.Linear(128, 256)
        self.anti_hafal_final = nn.Dropout(p=0.5)
        self.otak_final = nn.Linear(256, 10)

    def forward(self, x):
        # Eksekusi Blok 1
        x = F.relu(self.bn1a(self.mata1a(x)))
        x = F.relu(self.bn1b(self.mata1b(x)))
        x = self.kacamata_pengecil1(x)
        x = self.anti_hafal1(x)

        # Eksekusi Blok 2
        x = F.relu(self.bn2a(self.mata2a(x)))
        x = F.relu(self.bn2b(self.mata2b(x)))
        x = self.kacamata_pengecil2(x)
        x = self.anti_hafal2(x)

        # Eksekusi Blok 3
        x = F.relu(self.bn3a(self.mata3a(x)))
        x = F.relu(self.bn3b(self.mata3b(x)))
        x = self.kacamata_pengecil3(x)
        x = self.anti_hafal3(x)

        # Eksekusi Otak Final
        x = self.pooling_final(x)
        x = torch.flatten(x, 1) # Meratakan hasil pooling (Batch, 128)
        x = F.relu(self.otak_utama(x))
        x = self.anti_hafal_final(x)
        x = self.otak_final(x)
        return x

# ==========================================
# 3. DEKLARASI VARIABEL, GURU, & MEKANIK
# ==========================================
model = Otak().to(device)
guru_penilai = nn.CrossEntropyLoss()

mekanik_otak = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
pengatur_kecepatan = torch.optim.lr_scheduler.ReduceLROnPlateau(mekanik_otak, mode='max', factor=0.5, patience=2)

In [4]:
# ==========================================
# 4. FUNGSI BELAJAR & UJIAN
# ==========================================
def belajar(waktu_mulai):
    print("Mulai proses latihan...")
    model.train()

    for putaran in range(2):
        total_kesalahan = 0.0

        for indeks_suapan, data in enumerate(penyuap_latih, 0):
            gambar_masuk, kunci_jawaban = data

            gambar_masuk = gambar_masuk.to(device)
            kunci_jawaban = kunci_jawaban.to(device)

            mekanik_otak.zero_grad()
            tebakan = model(gambar_masuk)
            tingkat_kesalahan = guru_penilai(tebakan, kunci_jawaban)
            tingkat_kesalahan.backward()
            mekanik_otak.step()

            total_kesalahan += tingkat_kesalahan.item()

            if indeks_suapan % 500 == 499:
                waktu_berjalan = time.time() - waktu_mulai
                menit = int(waktu_berjalan // 60)
                detik = int(waktu_berjalan % 60)
                print(f'[Waktu: {menit}m {detik}s] Putaran {putaran + 1}, Data: {indeks_suapan + 1} | Kesalahan: {total_kesalahan / 500:.3f}')
                total_kesalahan = 0.0
    print("Sesi latihan selesai.")

def ujian():
    print("Memulai Ujian...")
    model.eval()
    jawaban_benar = 0
    total_soal = 0

    with torch.no_grad():
        for data in penyuap_uji:
            gambar_masuk, kunci_jawaban = data

            gambar_masuk = gambar_masuk.to(device)
            kunci_jawaban = kunci_jawaban.to(device)

            tebakan = model(gambar_masuk)
            _, tebakan_final = torch.max(tebakan.data, 1)
            total_soal += kunci_jawaban.size(0)
            jawaban_benar += (tebakan_final == kunci_jawaban).sum().item()

    akurasi = 100.0 * jawaban_benar / total_soal
    print(f'Akurasi saat ini: {akurasi:.2f}%')
    return akurasi

# ==========================================
# 5. LOGIKA UTAMA (TRAINING MANAGER DENGAN ROLLBACK)
# ==========================================
# Ditingkatkan menjadi 85% karena otak model sekarang jauh lebih kuat
target_akurasi = 85.0
akurasi_sekarang = 0.0
akurasi_terbaik = 0.0
sesi_ke = 1

waktu_mulai_global = time.time()

torch.save(model.state_dict(), './bobot_terbaik_temp.pth')

while akurasi_sekarang < target_akurasi:
    print(f"\n--- Memulai Sesi ke-{sesi_ke} (Target: {target_akurasi}%) ---")

    belajar(waktu_mulai_global)
    akurasi_sekarang = ujian()

    pengatur_kecepatan.step(akurasi_sekarang)

    if akurasi_sekarang > akurasi_terbaik:
        akurasi_terbaik = akurasi_sekarang
        torch.save(model.state_dict(), './bobot_terbaik_temp.pth')
        print(f"Akurasi meningkat menjadi {akurasi_terbaik:.2f}%.")
    else:
        print(f"Akurasi menurun ({akurasi_sekarang:.2f}% <= Rekor Terbaik: {akurasi_terbaik:.2f}%).")
        print("Mengembalikan model ke ingatan terbaik sebelumnya...")
        bobot_terbaik = torch.load('./bobot_terbaik_temp.pth', map_location=device)
        model.load_state_dict(bobot_terbaik)

    if akurasi_sekarang < target_akurasi:
        sesi_ke += 1
    else:
        waktu_total = time.time() - waktu_mulai_global
        menit_total = int(waktu_total // 60)
        print(f"\n Target {target_akurasi}% tercapai dalam waktu {menit_total} menit.")

        torch.save(model.state_dict(), './otak.pth')
        print("Otak final berhasil disimpan ke 'otak.pth'")

        if os.path.exists('./bobot_terbaik_temp.pth'):
            os.remove('./bobot_terbaik_temp.pth')


--- Memulai Sesi ke-1 (Target: 85.0%) ---
Mulai proses latihan...
[Waktu: 2m 4s] Putaran 1, Data: 500 | Kesalahan: 1.782
[Waktu: 4m 6s] Putaran 1, Data: 1000 | Kesalahan: 1.489
[Waktu: 6m 8s] Putaran 1, Data: 1500 | Kesalahan: 1.360
[Waktu: 8m 28s] Putaran 2, Data: 500 | Kesalahan: 1.277
[Waktu: 10m 30s] Putaran 2, Data: 1000 | Kesalahan: 1.220
[Waktu: 12m 32s] Putaran 2, Data: 1500 | Kesalahan: 1.177
Sesi latihan selesai.
Memulai Ujian...
Akurasi saat ini: 62.59%
Akurasi meningkat menjadi 62.59%.

--- Memulai Sesi ke-2 (Target: 85.0%) ---
Mulai proses latihan...
[Waktu: 15m 15s] Putaran 1, Data: 500 | Kesalahan: 1.136
[Waktu: 17m 17s] Putaran 1, Data: 1000 | Kesalahan: 1.109
[Waktu: 19m 19s] Putaran 1, Data: 1500 | Kesalahan: 1.067
[Waktu: 21m 38s] Putaran 2, Data: 500 | Kesalahan: 1.038
[Waktu: 23m 40s] Putaran 2, Data: 1000 | Kesalahan: 1.031
[Waktu: 25m 43s] Putaran 2, Data: 1500 | Kesalahan: 0.993
Sesi latihan selesai.
Memulai Ujian...
Akurasi saat ini: 69.26%
Akurasi meningkat m

In [9]:
# ==========================================
# 6. EVALUASI MODEL & EKSTRAK ERROR KE ZIP
# ==========================================
# Perbaikan Utama: Menggunakan Otak
model_tes = Otak().to(device)

try:
    bobot_disimpan = torch.load('./otak.pth', map_location=device)
    model_tes.load_state_dict(bobot_disimpan)
    model_tes.eval()

    daftar_label = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

    hitung_total = {label: 0 for label in daftar_label}
    hitung_benar = {label: 0 for label in daftar_label}
    hitung_salah = {label: 0 for label in daftar_label}

    jawaban_benar_global = 0
    total_soal_global = 0
    indeks_file_global = 0
    nama_file_zip_salah = "kumpulan_tebakan_salah.zip"

    # Alat bantu untuk mengubah tensor matematika kembali menjadi gambar visual
    tensor_ke_gambar = transforms.ToPILImage()

    with torch.no_grad(), zipfile.ZipFile(nama_file_zip_salah, 'w') as zip_salah:
        print(f"Menganalisis gambar dan membungkus tebakan yang salah ke '{nama_file_zip_salah}'...")

        for data in penyuap_uji_baru:
            gambar_masuk, kunci_jawaban = data

            gambar_masuk = gambar_masuk.to(device)
            kunci_jawaban = kunci_jawaban.to(device)

            tebakan = model_tes(gambar_masuk)
            _, tebakan_final = torch.max(tebakan.data, 1)

            total_soal_global += kunci_jawaban.size(0)
            jawaban_benar_global += (tebakan_final == kunci_jawaban).sum().item()

            for i in range(gambar_masuk.size(0)):
                prediksi_angka = tebakan_final[i].item()
                jawaban_angka = kunci_jawaban[i].item()

                label_asli = daftar_label[jawaban_angka]
                label_tebakan = daftar_label[prediksi_angka]

                hitung_total[label_asli] += 1

                if prediksi_angka == jawaban_angka:
                    hitung_benar[label_asli] += 1
                else:
                    hitung_salah[label_asli] += 1

                    # --- PERBAIKAN LOGIKA EKSTRAK KE ZIP ---

                    # 1. Ambil tensor gambar yang salah dan tarik kembali ke CPU
                    tensor_salah = gambar_masuk[i].cpu()

                    # 2. Denormalisasi: Mengembalikan warna asli (membalik rumus (x - 0.5) / 0.5)
                    tensor_salah = tensor_salah * 0.5 + 0.5

                    # 3. Ubah tensor menjadi format gambar PIL (Python Imaging Library)
                    gambar_pil = tensor_ke_gambar(tensor_salah)

                    # 4. Buat buffer memori virtual untuk menyimpan gambar sementara
                    buffer_memori = io.BytesIO()
                    gambar_pil.save(buffer_memori, format="PNG")

                    # 5. Susun nama file di dalam ZIP (karena nama file asli tidak ada, kita pakai indeks)
                    nama_di_dalam_zip = f"seharusnya_{label_asli}/ditebak_{label_tebakan}_urutan_ke_{indeks_file_global}.png"

                    # 6. Tulis data dari buffer memori langsung ke dalam file ZIP
                    zip_salah.writestr(nama_di_dalam_zip, buffer_memori.getvalue())

                indeks_file_global += 1

    # --- PERHITUNGAN AKURASI ---
    if total_soal_global > 0:
        akurasi_total = 100.0 * jawaban_benar_global / total_soal_global
    else:
        akurasi_total = 0.0

    print("\n=============================================")
    print(f"📊 SKOR AKURASI TOTAL: {akurasi_total:.2f}%")
    print("=============================================")
    print(f"Detail: Berhasil menebak {jawaban_benar_global} dari {total_soal_global} total gambar.")
    print(f"📁 Semua {total_soal_global - jawaban_benar_global} gambar yang salah telah dibungkus ke dalam '{nama_file_zip_salah}'.")

    print("\n📋 RINGKASAN EVALUASI JAWABAN PER KATEGORI")
    print("-" * 65)
    print(f"{'Kategori Objek':<15} | {'Total Gambar':<12} | {'Akurasi Benar %':<15}")
    print("-" * 65)

    for label in daftar_label:
        if hitung_total[label] > 0:
            rasio_akurasi = hitung_benar[label] / hitung_total[label]
        else:
            rasio_akurasi = 0.0

        print(f"{label:<15} | {hitung_total[label]:<12} | {rasio_akurasi:<15.2%}")

    print("-" * 65)

except FileNotFoundError:
    print("\nGagal! File 'otak.pth' tidak ditemukan di folder kerja saat ini.")

Menganalisis gambar dan membungkus tebakan yang salah ke 'kumpulan_tebakan_salah.zip'...

📊 SKOR AKURASI TOTAL: 85.55%
Detail: Berhasil menebak 8555 dari 10000 total gambar.
📁 Semua 1445 gambar yang salah telah dibungkus ke dalam 'kumpulan_tebakan_salah.zip'.

📋 RINGKASAN EVALUASI JAWABAN PER KATEGORI
-----------------------------------------------------------------
Kategori Objek  | Total Gambar | Akurasi Benar %
-----------------------------------------------------------------
airplane        | 1000         | 85.00%         
automobile      | 1000         | 95.00%         
bird            | 1000         | 76.20%         
cat             | 1000         | 63.20%         
deer            | 1000         | 89.00%         
dog             | 1000         | 84.00%         
frog            | 1000         | 90.80%         
horse           | 1000         | 87.80%         
ship            | 1000         | 91.90%         
truck           | 1000         | 92.60%         
--------------------------